# Part Of Speech Tagger

This notebook will develop a Part-of-Speech (POS) tagging system, leveraging the extensive Wall Street Journal (WSJ) corpus and the richly annotated Treebank dataset. It will aim to accurately categorize words into their respective POS tags, a fundamental task in the field of Natural Language Processing that serves as the foundation for numerous other language understanding applications. Various methods will be explored and applied to build a model that parses text to identify grammatical structures, providing insights into the syntactic framework of English as represented in the WSJ corpus.

In [1]:
import nltk
from nltk.corpus import treebank
from collections import defaultdict, Counter
import random
import numpy as np
nltk.download('treebank')

[nltk_data] Downloading package treebank to
[nltk_data]   Package treebank is already up-to-date!


True

Let's first collect the dataset:

In [2]:
# sentences = treebank.sents()
tagged_sentences = treebank.tagged_sents()
tagged_sentences_train = tagged_sentences[:-400]
tagged_sentences_test = tagged_sentences[-400:]

Run this cell to print examples of tagged sentences: 

In [3]:
for sentence in tagged_sentences_train[:5]:  # Adjust the number to print more or fewer sentences
    for word in sentence:
        print(f'{word[0]}/{word[1]}', end=' ')
    print()

Pierre/NNP Vinken/NNP ,/, 61/CD years/NNS old/JJ ,/, will/MD join/VB the/DT board/NN as/IN a/DT nonexecutive/JJ director/NN Nov./NNP 29/CD ./. 
Mr./NNP Vinken/NNP is/VBZ chairman/NN of/IN Elsevier/NNP N.V./NNP ,/, the/DT Dutch/NNP publishing/VBG group/NN ./. 
Rudolph/NNP Agnew/NNP ,/, 55/CD years/NNS old/JJ and/CC former/JJ chairman/NN of/IN Consolidated/NNP Gold/NNP Fields/NNP PLC/NNP ,/, was/VBD named/VBN *-1/-NONE- a/DT nonexecutive/JJ director/NN of/IN this/DT British/JJ industrial/JJ conglomerate/NN ./. 
A/DT form/NN of/IN asbestos/NN once/RB used/VBN */-NONE- */-NONE- to/TO make/VB Kent/NNP cigarette/NN filters/NNS has/VBZ caused/VBN a/DT high/JJ percentage/NN of/IN cancer/NN deaths/NNS among/IN a/DT group/NN of/IN workers/NNS exposed/VBN */-NONE- to/TO it/PRP more/RBR than/IN 30/CD years/NNS ago/IN ,/, researchers/NNS reported/VBD 0/-NONE- *T*-1/-NONE- ./. 
The/DT asbestos/NN fiber/NN ,/, crocidolite/NN ,/, is/VBZ unusually/RB resilient/JJ once/IN it/PRP enters/VBZ the/DT lungs/

Let's define a few variables that we will be using:

In [4]:
# Define special tags for the start and end of a sentence
start_sentence_tag = '<s>'
end_sentence_tag = '<e>'

# Initialize a set to store unique tags, including special tags for sentence boundaries
tags_set = set()
tags_set.add(start_sentence_tag)
tags_set.add(end_sentence_tag)

# Initialize a set to store the vocabulary (unique words from the sentences)
vocab = set()

# Iterate over each sentence in the corpus of tagged sentences
for sentence in tagged_sentences_train:
    # Iterate through each word-tag pair in the sentence
    for word, tag in sentence:
        # Add the tag to the set of tags
        tags_set.add(tag)
        # Add the word to the vocabulary set
        vocab.add(word)

# Calculate the size of the vocabulary
vocab_size = len(vocab)


# Hidden Markov Model (HMM) for POS

A Hidden Markov Model (HMM) is a statistical model that assumes the system being modeled is a Markov process with unobserved (hidden) states. HMMs are especially known for their application in temporal pattern recognition such as speech, handwriting, gesture recognition, part-of-speech tagging, and bioinformatics. The model is composed of hidden states, observable outputs, and the probabilities of transitions between these states and of generating the outputs from these states. This framework is particularly powerful for modeling sequences where the true states are not directly observable but can be inferred through observed data linked to these states.




## Building the transition and emission probability matrices

In this section, we'll focus on constructing the fundamental components of our POS tagging system: 

- `Matrix A`: The transition `Matrix A` will encapsulate the probabilities of moving from one POS tag to another,
- `Matrix B`: The emission `Matrix B` will detail the likelihood of a particular word being assigned a specific POS tag.

These matrices are key in predicting the sequence of POS tags in natural language texts. 
By building these matrices, we establish the groundwork for our tagging system using the WSJ corpus and Treebank dataset.



![.](hmm.png)
*An illustration of the two parts of an HMM representation: the A transition probabilities used to compute the prior probability, and the B observation likelihoods that are associated with each state, one likelihood for each possible observation word.*




### Statistical Foundations for HMM POS Tagging

To prepare for training our Hidden Markov Model (HMM) for POS tagging, we will commence by gathering essential statistics. Our objective is first to construct a trio of dictionaries—`tag_counts`,`tag_word_counts`, and `transition_counts`. These will respectively catalog the occurrences of the tags themselves, tally the frequency of each word within specific tags, and record the frequency of sequential tag transitions. This statistical foundation is critical for the effective training of our HMM.

In [5]:
# Define a dictrionary to store count of tags
tag_counts = {}
# Set the count for the start-of-sentence tag equal to the number of training sentences
# This assumes that each sentence in the training set begins with a specific start sentence tag
tag_counts[start_sentence_tag] = len(tagged_sentences_train)

# Set the count for the end-of-sentence tag equal to the number of training sentences
# This assumes that each sentence in the training set ends with a specific end sentence tag
tag_counts[end_sentence_tag] = len(tagged_sentences_train)

# Initialize a dictionary to store the counts of how often each word is tagged with a specific tag
tag_word_counts = {}

# Initialize a dictionary to store the counts of transitions from one tag to another
transition_counts = {}

## START YOUR CODE HERE
tag_word_counts = defaultdict(dict, tag_word_counts)
transition_counts = defaultdict(dict, transition_counts)
for sentence in tagged_sentences_train:
    prev = start_sentence_tag
    for word, tag in sentence:
        if tag in tag_counts:
            tag_counts[tag] += 1
        else:
            tag_counts[tag] = 1

        if tag not in tag_word_counts.keys():
            tag_word_counts[tag][word] = 1
        else:
            if word not in tag_word_counts[tag]:
                tag_word_counts[tag][word] = 1
            else:
                tag_word_counts[tag][word] += 1

        if prev not in transition_counts.keys():
            transition_counts[prev][tag] = 1
            prev = tag
            continue

        if tag not in transition_counts[prev]:
            transition_counts[prev][tag] = 1
        else:
            transition_counts[prev][tag] += 1
        prev = tag

    if prev not in transition_counts.keys():
        transition_counts[prev][end_sentence_tag] = 1
    else:
        if end_sentence_tag not in transition_counts[prev]:
            transition_counts[prev][end_sentence_tag] = 1
        else:
            transition_counts[prev][end_sentence_tag] += 1
            
## END

Run the code below to test your code:


In [6]:
for tag in tags_set:
    if tag == end_sentence_tag:
        continue
    assert tag_counts[tag] == np.sum(list(value for _, value in transition_counts[tag].items() )), 'Test failed'

assert tag_counts['<s>'] == 3514, 'Test failed'
print('Test successful')
assert tag_counts['NNP'] == 8468, 'Test failed'
print('Test successful')
assert tag_counts['NN'] == 11708, 'Test failed'
print('Test successful')
assert tag_counts['VBD'] == 2696, 'Test failed'
print('Test successful')

assert tag_word_counts['NNP']['Treasury'] == 39, 'Test failed'
print('Test successful')
assert tag_word_counts['NNP']['Exchange'] == 43, 'Test failed'
print('Test successful')
assert tag_word_counts['NNP']['Board'] == 40, 'Test failed'
print('Test successful')

assert tag_word_counts['NN']['group'] == 40, 'Test failed'
print('Test successful')
assert tag_word_counts['NN']['investor'] == 36, 'Test failed'
print('Test successful')
assert tag_word_counts['NN']['stock'] == 124, 'Test failed'
print('Test successful')

assert tag_word_counts['VBZ']['is'] == 603, 'Test failed'
print('Test successful')
assert tag_word_counts['VBZ']['has'] == 310, 'Test failed'
print('Test successful')
assert tag_word_counts['VBZ']['says'] == 208, 'Test failed'
print('Test successful')

assert transition_counts['NNP']['NNP'] == 3256, 'Test failed'
print('Test successful')
assert transition_counts['NNP']['NNS'] == 183, 'Test failed'
print('Test successful')
assert transition_counts['NNP']['WRB'] == 2, 'Test failed'
print('Test successful')

assert transition_counts['NN']['IN'] == 2872, 'Test failed'
print('Test successful')
assert transition_counts['NN']['PRP'] == 16, 'Test failed'
print('Test successful')
assert transition_counts['NN']['CC'] == 435, 'Test failed'
print('Test successful')


Test successful
Test successful
Test successful
Test successful
Test successful
Test successful
Test successful
Test successful
Test successful
Test successful
Test successful
Test successful
Test successful
Test successful
Test successful
Test successful
Test successful
Test successful
Test successful


### Transition matrix A

In the context of Hidden Markov Models (HMMs), the Transition Matrix $A$ contains the probabilities of moving from one state to another. Each element $a_{ij}$ of this matrix represents the probability of transitioning from state $i$ to state $j$. 
The matrix is essential for modeling the state dynamics in sequence prediction tasks, such as speech recognition or part-of-speech tagging, where the likelihood of moving between different states (e.g., from one part of speech to another) captures the structure of the data.


Formally, each transition probability value $a_{ij}$ is calculated as a **Laplace-smoothed conditional probability** $P(tag_j|tag_i)$ as follows:


$$a_{ij} = P(tag_j|tag_i) =\frac{c(tag_i,tag_{j})+1}{ c(tag_{i}) +\# Tags}$$


Here, $\#Tags$ represents the total number of unique tags, $c(tag_{i})$ indicates the frequency of occurrences of tag $i$ in the training set, and $c(tag_i, tag_{j})$ denotes how often tag $i$ and tag $j$ are observed consecutively in the dataset.
Overall, this approach adjusts the probabilities to ensure that no transition has a zero probability, even if it was not observed in the training data.

In [7]:
# Initialize a dictionary to store transition probabilities between tags
transition_probabilities = {}

## START YOUR CODE HERE
transition_probabilities = defaultdict(dict, transition_probabilities)
total_tags = len(tags_set)
for prev in tags_set:
    for cur in tags_set:
        transition_count = 0
        if prev in transition_counts:
            if cur in transition_counts[prev]:
                transition_count = transition_counts[prev][cur]
        
        transition_probabilities[prev][cur] = (transition_count + 1) / (tag_counts[prev] + total_tags)
## END

Rune this code to test your code:

In [8]:
for tag in tags_set:
    if tag == end_sentence_tag:
        continue
    assert abs(np.sum(list(transition_probabilities[tag].values())) - 1) <= 1e-5 


## Emission matrix B

In Hidden Markov Models (HMMs), the Emission Matrix $B$ describes the probabilities of observing various observable states (or observations) given each hidden state. Each element $b_{ij}$ in this matrix indicates the probability of observing the $j$-th observation from the 
$i$-th hidden state.

The emission matrix is critical for modeling how observed data, such as words in part-of-speech tagging or features in speech recognition, are generated from hidden states. It allows the model to associate observable evidence with appropriate hidden states, facilitating accurate inference and prediction.

Formally, each emission probability value $b_{ij}$ is calculated using a **Laplace-smoothed conditional probability** $P(w_j|tag_i)$, which represents the likelihood of observing word $w_j$ given the tag $i$. It is calculated as follows:

$$b_{ij} = P(w_j|tag_i) =\frac{c(w_j,tag_i)+1}{ c(tag_{i}) +|V|}$$

Here, $|V|$ represents the size of the vocabulary, $c(tag_{i})$ indicates the frequency of occurrences of tag $i$ in the training set, and $c(w_j,tag_i)$ denotes how often word $j$ was tagged with tag $i$ in the training dataset.
Similarly, this smoothing ensures that all potential observations have a non-zero probability of occurring, even if they weren't observed in the training data.

In [9]:
# Initialize a dictionary to store emission probabilities
emission_probabilities = {}

## START YOUR CODE HERE
emission_probabilities = defaultdict(dict, emission_probabilities)
vocab_len = len(vocab)
for tag in tags_set:
    for word in vocab:
        emission_count = 0
        if tag in tag_word_counts:
            if word in tag_word_counts[tag]:
                emission_count = tag_word_counts[tag][word]
        
        emission_probabilities[tag][word] = (emission_count + 1) / (tag_counts[tag] + vocab_len)
## END


Rune this code to test your code:

In [10]:
for tag in tags_set:
    if tag == start_sentence_tag or tag == end_sentence_tag:
        continue
    assert abs(np.sum(list(emission_probabilities[tag].values())) - 1) <= 1e-5 


## Decoding with Hidden Markov Models

Decoding is the process of determining the most probable sequence of hidden states (tags) based on observed data. Hence, given a sequence of words (observations) $W = w_1, w_2, \cdots, w_n$, decoding aims to find the most probable sequence of tags (states) $T = t_1, t_2, \cdots, t_n$ (each word in the input sequence is associated with a tag):

$$ \hat{t}_{1:n} = \underset{t_1, \cdots, t_n}{\operatorname{argmax}} P(t_1, \cdots, t_n|w_1, \cdots, w_n)$$

Using the Bayes' theorem, we obtain:

$$ \hat{t}_{1:n} = \underset{t_1, \cdots, t_n}{\operatorname{argmax}}  \frac{P(w_1, \cdots, w_n|t_1, \cdots, t_n)P(t_1, \cdots, t_n)}{P(w_1, \cdots, w_n)}$$

Since $P(w_1, \cdots, w_n)$ is constant for all possible tags, we can simplify this to:

$$ \hat{t}_{1:n} = \underset{t_1, \cdots, t_n}{\operatorname{argmax}}  P(w_1, \cdots, w_n|t_1, \cdots, t_n)P(t_1, \cdots, t_n)$$

HMM taggers make two further simplifying assumptions:
- The probability of a word appearing depends only on its own tag and is independent of neighboring words and tags:

$$P(w_1, \cdots, w_n|t_1, \cdots, t_n)\approx \prod_{i=1}^{n} P(w_i|t_i)$$

- The second assumption, the bigram assumption, is that the probability of a tag is dependent only on the previous tag, rather than the entire tag sequence:

$$P(t_1, \cdots, t_n)\approx \prod_{i=1}^{n} P(t_i|t_{i-1})$$


Plugging the simplifying assumptions results in the following equation for the most probable tag sequence from a bigram tagger:

$$ \hat{t}_{1:n} = \underset{t_1, \cdots, t_n}{\operatorname{argmax}}  \prod_{i=1}^{n}  \overbrace{P(w_i|t_i)}^\text{Emission B}    \underbrace{P(t_i|t_{i-1}}_\text{Transition A})$$

The two parts correspond neatly to the $B$ emission probability and $A$ transition probability that we defined previously!

## The Viterbi Algorithm
The Viterbi Algorithm is a dynamic programming algorithm used for finding the most likely sequence of hidden states—called the Viterbi path—that results in a sequence of observed events, especially in the context of Markov information sources and hidden Markov models (HMM). The algorithm was developed by Andrew Viterbi in 1967 and has since become widely used in various fields such as speech recognition, bioinformatics (notably in the decoding of genes in a DNA sequence), and communications.

The core idea behind the Viterbi Algorithm is to use a recursive approach to eliminate the less likely paths through the state space at each step of the computation, thus reducing the computational burden that would be involved in considering all paths. At each time step, it keeps track of the most probable path leading to each state and continually updates these paths based on new observations and the transitional probabilities between states.

The Viterbi algorithm first sets up a probability matrix or lattice where:
- Columns are observations (words of a sentence in the same sequence as in sentence).
- Rows as hidden states (all possible POS Tags are known).

Each cell of the matrix is represented by $V_t(j)$ (Viterbi value for t: column, j: row) having the probability that the HMM is in state $j$
(present POS Tag) after seeing the first $t$ observations (past words for which matrix (cell) values has been calculated) and passing
through the most probable state sequence (previous POS Tag) $t_1,\cdots,t_n$. Specifically, each Viterbi value $V_t(j)$ is computed by recursively taking the most probable path that could lead us to this cell as follows:


$$  V_t(j)= \underset{i=1}{\overset{n}{\operatorname{max}}} V_{t-1}(j) \times a_{i,j} \times b_j(w_t)$$

where:
- $V_{t-1}(j)$ is the previous Viterbi path probability from the previous time step
- $a_{i,j}$ is the transition probability from previous tag $t_i$ to current tag $t_j$
- $b_j(w_t)$ is the state observation likelihood of the observation word $w_t$ given the current state $j$

In [11]:
def viterbi(obs, states, trans_p, emit_p):
    """
    This function implements the Viterbi algorithm to find the most likely sequence of hidden states
    given a sequence of observations.

    Args:
        obs (list): A list of observations.
        states (set): A set of possible hidden states (excluding special start and end states).
        trans_p (dict): A dictionary where keys are source states and values are dictionaries representing
                        transition probabilities to other states.
        emit_p (dict): A dictionary where keys are hidden states and values are dictionaries representing
                       emission probabilities for each observation symbol.

    Returns:
        tuple: A tuple containing the probability of the most likely sequence and the most likely sequence itself (as a list).
    """

    # Remove special start and end states from the set of states
    states.discard('<s>')
    states.discard('<e>')

    # Initialize Viterbi variables
    V = [{}]  # Stores probability of the most likely sequence ending in each state at each time step
    path = {}  # Stores the best path (sequence of states) ending in each state at each time step
    
    ## START YOUR CODE HERE
    first_word = obs[0]
    num_observations = len(obs)
    
    for state in states:
        emission_prob = 0
        if first_word in emit_p[state]:
            emission_prob = emit_p[state][first_word]
        else:
            emission_prob = 1 / (tag_counts[state] + vocab_len)
        V[0][state] = (trans_p[start_sentence_tag][state]) * (emission_prob)
        path[state] = [state]

    
    for obs_index in range(1, num_observations):
        best_path = {}
        V.append({})

        cur_obs = obs[obs_index]

        for cur_state in states:
            combos = []
            emission_prob = 0
            if cur_obs in emit_p[cur_state]:
                emission_prob = emit_p[cur_state][cur_obs]
            else:
                emission_prob = 1 / (tag_counts[cur_state] + vocab_len)
            
            for prev_state in states:
                prob = (V[obs_index-1][prev_state]) * (trans_p[prev_state][cur_state]) * (emission_prob)
                combos.append((prob, prev_state))

            (max_prob, max_prev_state) = max(combos)
            V[obs_index][cur_state] = max_prob
            best_path[cur_state] = path[max_prev_state] + [cur_state]

        path = best_path
                
            
    combos = []
    for state in states:
        combos.append((V[num_observations-1][state], state))

    (prob, state) = max(combos)
            
    return (prob, path[state])    
    ## END



# Example usage:
instance = [word for word, _ in tagged_sentences_test[1]]

# Apply the algorithm
result = viterbi(instance, tags_set, transition_probabilities, emission_probabilities)
print(result)
assert result[1] == ['NNP', 'NNP', 'VBZ', 'RB', 'VBG', 'VBN', '-NONE-', 'IN', 'NNS', '.'], 'Test failed'
print('Test successful')

(1.4409863782833813e-34, ['NNP', 'NNP', 'VBZ', 'RB', 'VBG', 'VBN', '-NONE-', 'IN', 'NNS', '.'])
Test successful


## Evaluation

Let's now evaluate the performance of our HMM-based POS tagger using accuracy.


In [12]:
def calculate_accuracy(predicted_tags, ground_truth_tags):
    # Ensure that both lists are of the same length to avoid errors in comparison
    if len(predicted_tags) != len(ground_truth_tags):
        raise ValueError("The length of predicted tags and ground truth tags must be the same")

    # Count the number of correct predictions by comparing each element
    correct_predictions = sum(p == gt for p, gt in zip(predicted_tags, ground_truth_tags))
    
    # Calculate accuracy as the ratio of correct predictions to the total number of predictions
    accuracy = correct_predictions / len(ground_truth_tags)
    
    return accuracy

def global_accuracy(tagged_sentences_test):
    # Initialize a list to store accuracy of each sentence
    list_accuracy = []
    
    # Loop over each sentence in the test dataset
    for tagged_sentence_test in tagged_sentences_test:
        # Extract words from the sentence (ignoring the tags)
        instance = [word for word, _ in tagged_sentence_test]
        
        # Extract ground truth tags from the sentence
        ground_truth_tags = [tags for _, tags in tagged_sentence_test]
        
        # Predict tags using the Viterbi algorithm
        predicted_tags = viterbi(instance, tags_set, transition_probabilities, emission_probabilities)    
        
        # Calculate the accuracy for the current sentence
        accuracy = calculate_accuracy(predicted_tags[1], ground_truth_tags)
        
        # Append the calculated accuracy to the list
        list_accuracy.append(accuracy)
    
    # Calculate and return the mean of all accuracies
    return np.mean(list_accuracy)

# Execute the function on test data and print the global accuracy
global_accuracy = global_accuracy(tagged_sentences_test) 
print(f"Accuracy: {global_accuracy:.4f}")

# Assert statement to check if the computed accuracy matches the expected value
assert f'{global_accuracy:.4f}' == '0.8526', 'Test failed'


Accuracy: 0.8529


# Conditional Random Field for POS

Conditional Random Fields (CRFs) are a type of statistical modeling method often used in pattern recognition and machine learning, particularly for structured prediction. Unlike models such as Hidden Markov Models (HMMs) that assume independence between observed sequences, CRFs model the conditional probability of the output sequence given an input sequence, making them particularly well-suited for applications like Part-of-Speech (POS) tagging where context is crucial.

Key Concepts of CRFs:
- **Graphical Model**: CRFs are a form of undirected graphical model or Markov random field.
- **Sequence Modeling**: In POS tagging, CRFs consider the entire sequence of words and their features to predict the sequence of tags, making it robust against the contextual dependencies and ambiguities in natural language.
- **Feature Functions**: They utilize feature functions that can be defined to capture various linguistic aspects such as word suffixes, prefixes, neighboring words, or even specific word-tag combinations. These features are not limited to the immediate neighbors, allowing broader context consideration.
- **Training Objective**: The training of a CRF involves adjusting the weights assigned to different features such that the predicted label sequence maximizes the conditional probability of the true label sequence given the observed data sequence.
- **Challenge**: Implementing a CRF for POS Tagging

We would implement a Conditional Random Field (CRF) to perform Part-of-Speech tagging. The goal is to accurately predict the POS tags for each word in a given sentence, considering the context provided by adjacent words and their features.

This task requires understanding both machine learning principles and natural language processing techniques. Next, we would implement and tune a CRF for POS tagging.



### Load libraries

In [13]:
from math import exp, log
import re
from scipy.optimize import fmin_l_bfgs_b

### Create Dataset reusing tagged_sentences_train & tagged_sentences_test

In [14]:
nltk_training_data = []
nltk_testing_data = []

for sentence in tagged_sentences_train: 
    curr_sent = []
    curr_labels = []
    for word, label in sentence:
        curr_sent.append(word)
        curr_labels.append(label)
    training_data_pair = (curr_sent, curr_labels)
    nltk_training_data.append(training_data_pair)

for sentence in tagged_sentences_test: 
    curr_sent = []
    curr_labels = []
    for word, label in sentence:
        curr_sent.append(word)
        curr_labels.append(label)
    testing_data_pair = (curr_sent, curr_labels)
    nltk_testing_data.append(testing_data_pair)

### Create Feature templates

In [15]:
def get_feature_templates(X, t):
    
    length = len(X)
    comparators = {'more', 'less', 'earlier', 'later', 'better'}
    pronouns = {'i', 'he', 'she', 'they', 'him', 'it', 'you', 'we', 'us'}
    possessive_pronouns = {'my', 'his', 'her', 'their', 'our', 'your', 'its'}
    determiners = {'a', 'an', 'the', 'this', 'these'}
    verb_past = {'was', 'said', 'did', 'had', 'were'}
    numeric_values = {'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 
                      'nine', 'ten', 'hundred', 'hundreds', 'thousand', 'thousands', 'million', 'billion'}
    modals = {'will', 'would', 'may', 'might', 'can', 'could'}
    preposition = {'of', 'as', 'that', 'with', 'in', 'by', 'for', 'on', 'than', 'from', 'because', 'over'}
    predetermined = {'all', 'such'}
    wh_pronoun = {'who', 'what', 'whom'}
    wh_adverb = {'when', 'how', 'why', 'where'}
    verb_3rd_person_sing_present = {'is', 'has'}
    rb_extras = {"n't", "not", "also"}

    features = list()
    features.append('CUR_WORD:%s' % X[t])

    word = X[t]

    if word.endswith("ly") or word.endswith("ward") or word.endswith("wise") or word.lower() in rb_extras:
        features.append('IS_RB:%s' % X[t])

    if word.lower() in comparators:
        features.append('IS_RBR:%s' % X[t])

    if word.lower() == "most":
        features.append('IS_RBS:%s' % X[t])
    
    if word.lower() in pronouns:
        features.append('IS_PRP:%s' % X[t])
    
    if word.lower() in possessive_pronouns:
        features.append('IS_PRP$:%s' % X[t])
    
    if word.lower() in verb_3rd_person_sing_present:
        features.append('IS_VBZ:%s' % X[t])
    
    if word.endswith("ify") or word.endswith("ize"):
        features.append('IS_VB:%s' % X[t])
    
    if word.endswith("ing"):
        features.append('IS_VBG:%s' % X[t])
    
    if word.lower() in verb_past:
        features.append('IS_VBD:%s' % X[t])
    
    if word.lower() == "been":
        features.append('IS_VBN:%s' % X[t])
    
    if word.endswith("able") or word.endswith("ible") or word.endswith("ful") or word.endswith("ic"):
        features.append('IS_JJ:%s' % X[t])
    
    if word.endswith("age") or word.endswith("ness") or word.endswith("ship") or word.endswith("tion") \
                                    or word.endswith("ity") or word.endswith("ure") or word == '%':
        features.append('IS_NN:%s' % X[t])
    
    if word.lower() in determiners:
        features.append('IS_DT:%s' % X[t])
    
    if word.lower() in numeric_values or re.match('^[-+]?[0-9]*\.?[0-9]+$', word):
        features.append('IS_CD:%s' % X[t])
    
    if word.lower() in modals:
        features.append('IS_MD:%s' % X[t])
    
    if word[0].isupper() and word not in determiners and word not in modals and word not in pronouns:
        features.append('IS_NNP:%s' % X[t])
    
    if word[0].isupper() and word.endswith('s') and word not in determiners and word not in modals and word not in pronouns:
        features.append('IS_NNPS:%s' % X[t])
    
    if word in preposition:
        features.append('IS_IN:%s' % X[t])
    
    if word.lower() in predetermined:
        features.append('IS_PDT:%s' % X[t])
    
    if word.lower() == "to":
        features.append('IS_TO:%s' % X[t])
    
    if word.lower() == "$":
        features.append('IS_DOLLAR:%s' % X[t])
    
    if word == "'s":
        features.append('IS_POS:%s' % X[t])
    
    if word.lower() in wh_pronoun:
        features.append('IS_WP:%s' % X[t])
    
    if word.lower() == "whose":
        features.append('IS_WP:%s' % X[t])
    
    if word.lower() == "which":
        features.append('IS_WDT:%s' % X[t])
    
    if word.lower() in wh_adverb:
        features.append('IS_WRB:%s' % X[t])
    
    
    if t < length-1:
        features.append('NEXT_WORD:%s' % (X[t+1]))
        features.append('CUR_NEXT_WORD:%s %s' % (X[t], X[t+1]))
        
        if t < length-2:
            features.append('WORD_AFTER_NEXT:%s' % (X[t+2]))
            
    if t > 0:
        features.append('PREV_WORD:%s' % (X[t-1]))
        features.append('PREV_CUR_WORD:%s %s' % (X[t-1], X[t]))
        
        if t > 1:
            features.append('WORD_BEFORE_PREV:%s' % (X[t-2]))
            

    return features

### Construct the Features and Labels

In [16]:
# setting the label for the starting label, i.e., t = -1
start_label = '*START'
start_label_index = 0

feature_dic = dict()
observation_set = set()
emp_counter = Counter()
num_features = 0

label_dic = {start_label: start_label_index}
label_array = [start_label]

feature_templates = get_feature_templates

# Creates features and labels from input data
def create_features_labels(data):
    for X, Y in data:
        prev_y = start_label_index
        for t in range(len(X)):
            # Gets a label id
            if Y[t] not in label_dic:
                y = len(label_dic)
                label_dic[Y[t]] = y
                label_array.append(Y[t])
            else:
                y = label_dic[Y[t]]
                
            # Adds features
            add_features(prev_y, y, X, t)
            prev_y = y

    return label_dic, label_array

# Generates features, constructs feature_dic
def add_features(prev_y, y, X, t):
    # declared with global scope to handle UnboundLocalError
    global num_features
    for feature_string in feature_templates(X, t):
        if feature_string not in feature_dic.keys():
            feature_dic[feature_string] = dict()
            
            # Bigram feature
            feature_id = num_features
            feature_dic[feature_string][(prev_y, y)] = feature_id
            emp_counter[feature_id] += 1
            num_features += 1
            
            # Unigram feature
            feature_id = num_features
            feature_dic[feature_string][(-1, y)] = feature_id
            emp_counter[feature_id] += 1
            num_features += 1
        else:
            if (prev_y, y) in feature_dic[feature_string].keys():
                emp_counter[feature_dic[feature_string][(prev_y, y)]] += 1
            else:
                feature_id = num_features
                feature_dic[feature_string][(prev_y, y)] = feature_id
                emp_counter[feature_id] += 1
                num_features += 1
            if (-1, y) in feature_dic[feature_string].keys():
                emp_counter[feature_dic[feature_string][(-1, y)]] += 1
            else:
                feature_id = num_features
                feature_dic[feature_string][(-1, y)] = feature_id
                emp_counter[feature_id] += 1
                num_features += 1

# Gets a list of feature ids of given observation and transition.
def get_feature_vector(prev_y, y, X, t):
    feature_ids = list()
    for feature_string in feature_templates(X, t):
        try:
            feature_ids.append(feature_dic[feature_string][(prev_y, y)])
        except KeyError:
            pass
    return feature_ids

# Calculates inner products of the given parameters and feature vectors of the given observations at time t
def calc_inner_products(params, X, t):
    inner_products = Counter()
    for feature_string in feature_templates(X, t):
        try:
            for (prev_y, y), feature_id in feature_dic[feature_string].items():
                inner_products[(prev_y, y)] += params[feature_id]
        except KeyError:
            pass
    return [((prev_y, y), score) for (prev_y, y), score in inner_products.items()]

def get_empirical_counts():
    empirical_counts = np.ndarray((num_features,))
    for feature_id, counts in emp_counter.items():
        empirical_counts[feature_id] = counts
    return empirical_counts

def get_feature_list(X, t):
    feature_list_dic = dict()
    for feature_string in feature_templates(X, t):
        for (prev_y, y), feature_id in feature_dic[feature_string].items():
            if (prev_y, y) in feature_list_dic.keys():
                feature_list_dic[(prev_y, y)].add(feature_id)
            else:
                feature_list_dic[(prev_y, y)] = {feature_id}
    return [((prev_y, y), feature_ids) for (prev_y, y), feature_ids in feature_list_dic.items()]

### Compute the log likelihood

In [17]:
num_iteration = 0
num_sub_iteration = 0
total_sub_iterations = 0
GRADIENT = None


def _update_iter_count(params):
    # declared with global scope to handle UnboundLocalError
    global num_iteration
    global num_sub_iteration
    global total_sub_iterations
    num_iteration += 1
    total_sub_iterations += num_sub_iteration
    num_sub_iteration = 0

def _gen_potential_table_for_training(params, num_labels, X):
    tables = list()
    for t in range(len(X)):
        table = np.zeros((num_labels, num_labels))
        for (prev_y, y), feature_ids in X[t]:
            score = sum(params[fid] for fid in feature_ids)
            if prev_y == -1:
                table[:, y] += score
            else:
                table[prev_y, y] += score
        table = np.exp(table)
        if t == 0:
            table[start_label_index+1:] = 0
        else:
            table[:,start_label_index] = 0
            table[start_label_index,:] = 0
        tables.append(table)

    return tables

def _gen_potential_table_for_inference(params, num_labels, X):
    tables = list()
    for t in range(len(X)):
        table = np.zeros((num_labels, num_labels))
        for (prev_y, y), score in calc_inner_products(params, X, t):
            if prev_y == -1:
                table[:, y] += score
            else:
                table[prev_y, y] += score
        
        table = np.exp(table)
        if t == 0:
            table[start_label_index+1:] = 0
        else:
            table[:,start_label_index] = 0
            table[start_label_index,:] = 0
        tables.append(table)

    return tables


def _calc_forward_backward_Z(num_labels, time_length, potential_table):
    # forward terms
    alpha = np.zeros((time_length, num_labels))
    scaling_dic = dict()
    t = 0
    for label_id in range(num_labels):
        alpha[t, label_id] = potential_table[t][start_label_index, label_id]
    
    t = 1
    while t < time_length:
        scaling_time = None
        scaling_coefficient = None
        overflow_occured = False
        label_id = 1
        while label_id < num_labels:
            alpha[t, label_id] = np.dot(alpha[t-1,:], potential_table[t][:,label_id])
            if alpha[t, label_id] > 1e250:
                if overflow_occured:
                    raise BaseException()
                overflow_occured = True
                scaling_time = t - 1
                scaling_coefficient = 1e250
                scaling_dic[scaling_time] = scaling_coefficient
                break
            else:
                label_id += 1
        if overflow_occured:
            alpha[t-1] /= scaling_coefficient
            alpha[t] = 0
        else:
            t += 1

    # backward terms
    beta = np.zeros((time_length, num_labels))
    t = time_length - 1
    for label_id in range(num_labels):
        beta[t, label_id] = 1.0
    #beta[time_length - 1, :] = 1.0     # slow
    for t in range(time_length-2, -1, -1):
        for label_id in range(1, num_labels):
            beta[t, label_id] = np.dot(beta[t+1,:], potential_table[t+1][label_id,:])
        if t in scaling_dic.keys():
            beta[t] /= scaling_dic[t]

    # instance-specific normalization factor
    Z = sum(alpha[time_length-1])

    return alpha, beta, Z, scaling_dic


# Calculate likelihood and gradient
def _calc_log_likelihood_gradient(params, training_data, training_feature_data, empirical_counts, label_dic):
    
    squared_sigma = 10.0
    expected_counts = np.zeros(num_features)

    total_logZ = 0
    for X_features in training_feature_data:
        potential_table = _gen_potential_table_for_training(params, len(label_dic), X_features)
        alpha, beta, Z, scaling_dic = _calc_forward_backward_Z(len(label_dic), len(X_features), potential_table)
        
        
        total_logZ += log(Z) + sum(log(scaling_coefficient) for _, scaling_coefficient in scaling_dic.items())
        
        for t in range(len(X_features)):
            potential = potential_table[t]
            for (prev_y, y), feature_ids in X_features[t]:
                # Adds p(prev_y, y | X, t)
                if prev_y == -1:
                    if t in scaling_dic.keys():
                        prob = (alpha[t, y] * beta[t, y] * scaling_dic[t])/Z
                    else:
                        prob = (alpha[t, y] * beta[t, y])/Z
                elif t == 0:
                    if prev_y is not start_label_index:
                        continue
                    else:
                        prob = (potential[start_label_index, y] * beta[t, y])/Z
                else:
                    if prev_y is start_label_index or y is start_label_index:
                        continue
                    else:
                        prob = (alpha[t-1, prev_y] * potential[prev_y, y] * beta[t, y]) / Z
                for fid in feature_ids:
                    expected_counts[fid] += prob

    likelihood = np.dot(empirical_counts, params) - total_logZ - np.sum(np.dot(params,params))/(squared_sigma*2)

    gradients = empirical_counts - expected_counts - params/squared_sigma
    
    # declared with global scope to handle UnboundLocalError
    global GRADIENT
    GRADIENT = gradients
    global num_sub_iteration

    
    sub_iteration_str = '    '
    if num_sub_iteration > 0:
        sub_iteration_str = '(' + '{0:02d}'.format(num_sub_iteration) + ')'
    print('  ', '{0:03d}'.format(num_iteration), sub_iteration_str, ':', likelihood * -1)

    num_sub_iteration += 1

    return likelihood * -1


def _gradient(params, *args):
    return GRADIENT * -1

### Linear-chain Conditional Random Field

In [18]:
# Linear-chain Conditional Random Field
class LinearChainCRF:
    
    def __init__(self):
        self.params = None
        self.training_data = nltk_training_data
        self.label_dic = None
        self.label_array = None
        self.num_labels = 0


    def _get_training_feature_data(self):
        return [[get_feature_list(X, t) for t in range(len(X))]
                for X, _ in self.training_data]

    # Estimates parameters to maximize log-likelihood using 
    # using conjugate gradient methods - L-BFGS
    def _estimate_parameters(self):
        training_feature_data = self._get_training_feature_data()
        self.params, log_likelihood, information = \
                fmin_l_bfgs_b(func=_calc_log_likelihood_gradient, fprime=_gradient,
                              x0=np.zeros(num_features), 
                              args=(self.training_data, training_feature_data,
                                    get_empirical_counts(),
                                    self.label_dict),
                              callback=_update_iter_count)
        print('   ========================')
        print('   (iter: iteration, sit: sub iteration)')
        print('* Training has been finished with %d iterations' % information['nit'])

        if information['warnflag'] != 0:
            print('* Warning (code: %d)' % information['warnflag'])
            if 'task' in information.keys():
                print('* Reason: %s' % (information['task']))
        print('* Likelihood: %s' % str(log_likelihood))

    
    def start_training(self):
        print('Start training')
        self.label_dict, self.label_array = create_features_labels(self.training_data)
        self.num_labels = len(self.label_array)
        
        print("* Number of labels: %d" % (self.num_labels-1))
        print("* Number of features: %d" % num_features)

        self._estimate_parameters()      
        print('Training Complete')

    def test(self, testing_data):
        total_count = 0
        correct_count = 0
        for X, Y in testing_data:
            Yprime = self.inference(X)
            for t in range(len(Y)):
                total_count += 1
                if Y[t] == Yprime[t]:
                    correct_count += 1
        return (correct_count, total_count)


    # Gets the best label sequence
    def inference(self, X):
        self.inf_label_dic = {label: i for label, i in enumerate(label_array)}
        potential_table = _gen_potential_table_for_inference(self.params, self.num_labels, X)
        
        time_length = len(X)
        max_table = np.zeros((time_length, self.num_labels))
        argmax_table = np.zeros((time_length, self.num_labels), dtype='int64')
        
        # Using Viterbi with backtracking
        t = 0
        for label_id in range(self.num_labels):
            max_table[t, label_id] = potential_table[t][start_label_index, label_id]
        for t in range(1, time_length):
            for label_id in range(1, self.num_labels):
                max_value = -float('inf')
                max_label_id = None
                for prev_label_id in range(1, self.num_labels):
                    value = max_table[t-1, prev_label_id] * potential_table[t][prev_label_id, label_id]
                    if value > max_value:
                        max_value = value
                        max_label_id = prev_label_id
                max_table[t, label_id] = max_value
                argmax_table[t, label_id] = max_label_id

        sequence = list()
        next_label = max_table[time_length-1].argmax()
        sequence.append(next_label)
        for t in range(time_length-1, -1, -1):
            next_label = argmax_table[t, next_label]
            sequence.append(next_label)
        
        Yprime = [self.inf_label_dic[label_id] for label_id in sequence[::-1][1:]]
        return Yprime

### Instantiate and Train the Linear Chain CRF

In [19]:
crf = LinearChainCRF()

In [20]:
crf.start_training()

Start training
* Number of labels: 46
* Number of features: 537471
   000      : 347158.2514914585
   000 (01) : 333498.8480253227
   000 (02) : 280900.7240026074
   001      : 236607.31443465574
   002      : 185901.784681677
   003      : 132777.41689116484
   004      : 103934.58959218234
   005      : 89867.25331010736
   006      : 79553.43425865336
   007      : 70177.19801255368
   008      : 59761.256328310694
   009      : 49698.073708373035
   010      : 40001.098783289
   011      : 31610.466414345603
   012      : 26184.747155865105
   013      : 23273.463598054357
   014      : 18963.329442925944
   015      : 16127.206938820178
   016      : 13835.932243914975
   017      : 10801.132634403744
   018      : 8718.771894089063
   019      : 7800.772283288527
   020      : 6971.8917652497985
   021      : 5959.84391990986
   022      : 5255.328064667134
   023      : 4782.821903209679
   024      : 4491.629293240674
   025      : 4339.598688538066
   026      : 4116.486451655

### Test the Linear Chain CRF

In [21]:
correct_count, total_count = crf.test(nltk_testing_data)

### Performance Metrics

In [22]:
from prettytable import PrettyTable
metrics_table = PrettyTable()
metrics_table.field_names = ["Total Tags", "Correct Tags", "Performance"]
metrics_table.add_row([total_count, correct_count, (correct_count/total_count) * 100])
print(metrics_table)

+------------+--------------+-------------------+
| Total Tags | Correct Tags |    Performance    |
+------------+--------------+-------------------+
|   10002    |     9085     | 90.83183363327333 |
+------------+--------------+-------------------+
